# 05 — Weight-level learning, visually

Now replace a table or explicit `if/else` rules with a tiny neural network.

We will train it to imitate the readable Elder Mist policy first, because supervised learning makes the mechanics of backprop easiest to see.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Tiny neural policy</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">INPUT</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">W₁</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">HIDDEN UNITS</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">W₂</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">ACTION LOGITS</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))

from eldermist import ElderMistReplicaEnv, ACTIONS
from eldermist.policies import sensible_rule_policy
from eldermist.mlp_clone import TinyMLP

# One synthetic architecture picture as an actual figure.
plt.figure(figsize=(10,4))
for x, count, label in [(0.15,6,"input"),(0.50,8,"hidden"),(0.85,5,"actions")]:
    ys = np.linspace(0.15,0.85,count)
    plt.scatter([x]*count, ys, s=120)
    plt.text(x,0.97,label,ha="center",weight="bold")
for y1 in np.linspace(0.15,0.85,6):
    for y2 in np.linspace(0.15,0.85,8):
        plt.plot([0.15,0.50],[y1,y2],linewidth=0.25,alpha=0.25)
for y1 in np.linspace(0.15,0.85,8):
    for y2 in np.linspace(0.15,0.85,5):
        plt.plot([0.50,0.85],[y1,y2],linewidth=0.25,alpha=0.25)
plt.xlim(0,1); plt.ylim(0,1.05); plt.axis("off")
plt.title("A neural policy is a function with trainable numeric weights")
plt.show()

## 1. Collect demonstration pairs

Behavior cloning uses examples:

`observation → demonstrated action`.

This is not RL yet. There is no reward-driven update in this training objective.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Behavior-cloning dataset</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">OBSERVATION X</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">TEACHER POLICY</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">ACTION y</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
X, y = [], []
for seed in range(200):
    e = ElderMistReplicaEnv()
    o, _ = e.reset(seed=seed)
    while True:
        a = sensible_rule_policy(e, o)
        X.append(o.copy()); y.append(a)
        o, _, term, trunc, _ = e.step(a)
        if term or trunc: break

X = np.asarray(X, float)
y = np.asarray(y, int)

counts = pd.Series(y).value_counts().sort_index()
plt.figure(figsize=(9,4))
plt.bar([ACTIONS[i] for i in counts.index], counts.values)
plt.xticks(rotation=25)
plt.ylabel("Training examples")
plt.title("Actions present in the imitation dataset")
plt.show()

## 2. Before training, weights are mostly random

The matrix `W1` connects input dimensions to hidden units.

Those values are literally trainable parameters.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Weights create representation</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">INPUT DIMENSIONS</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">W₁ MATRIX</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">HIDDEN REPRESENTATION</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
model = TinyMLP(in_dim=X.shape[1], hidden_dim=32, out_dim=len(ACTIONS), seed=0)
before = model.W1.copy()

plt.figure(figsize=(10,5))
plt.imshow(before, aspect="auto")
plt.xlabel("Hidden unit")
plt.ylabel("Input feature")
plt.title("W1 before training")
plt.colorbar(label="weight")
plt.show()

## 3. Forward pass → loss → backward pass → weight update

Backpropagation computes how changing each weight would change the loss.

The optimizer then actually changes the numbers.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Backprop loop</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">FORWARD PASS</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">LOSS</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">GRADIENTS</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">UPDATE WEIGHTS</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(X))
split = int(0.8*len(X))
tr, te = perm[:split], perm[split:]

losses = []
accs = []

for epoch in range(120):
    loss = model.train_step(X[tr], y[tr], lr=0.05)
    losses.append(loss)
    pred = model.predict(X[te])
    accs.append((pred == y[te]).mean())

plt.figure(figsize=(9,4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training loss falls as weights change")
plt.show()

## 4. The weights really changed

This is different from merely changing a prompt or adding RAG context.

The numeric parameters inside the model are now different.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Actual parameter learning</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">W BEFORE</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">GRADIENT UPDATE</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">W AFTER</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
after = model.W1.copy()
delta = np.abs(after - before)

plt.figure(figsize=(10,5))
plt.imshow(delta, aspect="auto")
plt.xlabel("Hidden unit")
plt.ylabel("Input feature")
plt.title("Absolute change in W1 after training")
plt.colorbar(label="|Δ weight|")
plt.show()

print("Mean absolute weight change:", delta.mean())

## 5. What action does the trained network prefer?

The final layer produces logits, then softmax converts them to action probabilities.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Neural policy output</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">OBSERVATION</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">NETWORK</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">LOGITS</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">SOFTMAX</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">ACTION PROBS</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
sample = X[te[0:1]]
probs = model.predict_proba(sample)[0]

plt.figure(figsize=(9,4))
plt.bar(ACTIONS, probs)
plt.xticks(rotation=25)
plt.ylim(0,1)
plt.ylabel("Probability")
plt.title("Action probabilities from the trained tiny network")
plt.show()

## 6. Behavior cloning vs RL

Both can train neural weights.

The difference is the training signal:

- behavior cloning: imitate demonstrated actions
- RL: improve expected future reward from interaction


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Supervised signal</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">DEMONSTRATION LABEL</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">BEHAVIOR CLONING</text>
<line x1="180.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
methods = ["Behavior cloning","RL"]
signals = [1,1]
labels = ["expert action label","reward / return"]

plt.figure(figsize=(8,4))
plt.bar(methods, signals)
plt.ylim(0,1.3)
for i, lab in enumerate(labels):
    plt.text(i, 1.05, lab, ha="center")
plt.ylabel("Both can update weights")
plt.title("Same neural machinery, different source of learning signal")
plt.show()

## Once this clicks

PPO/OpenAI Five stops being “mysterious neural RL.”

It is a much larger policy/value network whose weights are repeatedly updated using reward-derived objectives from enormous amounts of experience.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Bridge to large-scale RL</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">EXPERIENCE</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">REWARD-DERIVED LOSS</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">BACKPROP</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">WEIGHTS</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">BETTER POLICY</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>
